# Sugar-Beets 2016：左右裁边 + 缩放至 350×350

- **输入**：`EWS/data/sugar-beets-2016-DatasetNinja/ds/img/` 下 `rgb_*.png`（**1296×966**，PIL 读入为 `size=(W,H)`）。
- **裁切**：左右各去掉 **165** 列 → 宽 `1296−330=966`，高仍为 **966**，得到 **966×966** 方形区域（列范围 `[165, 1131)`，行 `[0, 966)`）。
- **缩放**：双线性 / Lanczos 缩放到 **350×350** RGB。
- **输出**：默认写入同数据集下的 `ds/img_350_crop/`（与 `img` 并列），文件名与源图一致；若目录不存在会自动创建。

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
from PIL import Image

W0, H0 = 1296, 966
CROP_EACH_SIDE = 165
OUT_WH = (350, 350)

x0 = CROP_EACH_SIDE
x1 = W0 - CROP_EACH_SIDE
crop_w = x1 - x0
assert crop_w == H0 == 966, (crop_w, H0)

try:
    _RESIZE = Image.Resampling.LANCZOS
except AttributeError:
    _RESIZE = Image.LANCZOS

print(f"裁切框 (left, upper, right, lower) = ({x0}, 0, {x1}, {H0}) → {crop_w}×{H0}")
print(f"输出尺寸: {OUT_WH[0]}×{OUT_WH[1]}")

In [ ]:
def find_repo_with_sugar_img(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        img_dir = p / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img"
        if img_dir.is_dir():
            return p
    raise FileNotFoundError(
        "未找到 EWS/data/sugar-beets-2016-DatasetNinja/ds/img，请在仓库根目录启动内核。"
    )


REPO = find_repo_with_sugar_img(Path.cwd())
IMG_DIR = REPO / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img"
OUT_DIR = REPO / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img_350_crop"

#若希望强制使用固定绝对路径，可改为：
# IMG_DIR = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\sugar-beets-2016-DatasetNinja\ds\img")
# OUT_DIR = IMG_DIR.parent / "img_350_crop"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO:", REPO)
print("IMG_DIR:", IMG_DIR)
print("OUT_DIR:", OUT_DIR)

In [ ]:
def process_one(src: Path, dst: Path) -> None:
    im = Image.open(src).convert("RGB")
    if im.size != (W0, H0):
        raise ValueError(f"期望 {W0}×{H0}，得到 {im.size[0]}×{im.size[1]}: {src.name}")
    cropped = im.crop((x0, 0, x1, H0))
    if cropped.size != (crop_w, H0):
        raise RuntimeError(f"裁切后尺寸异常: {cropped.size}")
    out = cropped.resize(OUT_WH, _RESIZE)
    out.save(dst, format="PNG", optimize=True)


paths = sorted(IMG_DIR.glob("rgb_*.png"))
if not paths:
    raise FileNotFoundError(f"未找到 rgb_*.png: {IMG_DIR}")

print(f"待处理: {len(paths)} 张")

In [ ]:
from tqdm import tqdm

for p in tqdm(paths, desc="crop+resize"):
    process_one(p, OUT_DIR / p.name)

print(f"完成，输出目录: {OUT_DIR}")

In [ ]:
# 抽查第一张输出
sample = OUT_DIR / paths[0].name
chk = Image.open(sample)
arr = np.asarray(chk)
print(sample.name, "size", chk.size, "dtype", arr.dtype, "shape", arr.shape)